In [6]:
import numpy as np

TARGETS = ["Y"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [7]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [8]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
display(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====


,Target,Dataset,Best_Model,Neurons,Best_R2
0,Y,Train_1,model_32_Ld0.7_Lp0.3_r0.01_seed0,[32],[0.9839536697793403]
1,Y,Train_2,model_16_8_Ld0.7_Lp0.3_r0.9_seed1,"[16, 8]",[0.7081942251788943]
2,Y,Val,model_264_128_Ld0.3_Lp0.7_r0.9_seed2,"[264, 128]",[0.975812627130699]
3,Y,Test_1,model_264_128_Ld0.7_Lp0.3_r0.5_seed1,"[264, 128]",[0.9623620580899729]
4,Y,Test_2,model_8_4_Ld0.7_Lp0.3_r0.5_seed0,"[8, 4]",[0.7286890281357081]
5,Y,Test_3,model_32_Ld0.3_Lp0.7_r0.9_seed0,[32],[0.9124910241313141]


In [9]:
df_summary_top10["bestR2"] = best_only_df["Best_R2"].str.strip("[]").astype(float)
df_summary_top10["Neurons"] = best_only_df["Neurons"]
display(df_summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,Y,0.967756,0.006980,0.002702,0.000585,0.983954,[32]
1,Train_2,Y,0.705279,0.001391,0.031735,0.000150,0.708194,"[16, 8]"
2,Val,Y,0.962182,0.008021,0.002816,0.000597,0.975813,"[264, 128]"
3,Test_1,Y,0.956838,0.003291,0.003537,0.000270,0.962362,"[264, 128]"
4,Test_2,Y,0.721341,0.003858,0.029578,0.000410,0.728689,"[8, 4]"
5,Test_3,Y,0.905845,0.003954,0.007232,0.000304,0.912491,[32]


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted